In [1]:
import pandas as pd


In [2]:
DTYPES = {
    "Invoice": str,
    "StockCode": str,
}

In [3]:
from pathlib import Path
Path.cwd()

WindowsPath('c:/Users/OMISTAJA/Documents/repos/mini-data-pipeline/notebooks')

In [4]:
df = pd.read_csv(
    "../data/combined_retail_data.csv",
    dtype=DTYPES,
    parse_dates=["InvoiceDate"]
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1044848 entries, 0 to 1044847
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1044848 non-null  str           
 1   StockCode    1044848 non-null  str           
 2   Description  1040573 non-null  str           
 3   Quantity     1044848 non-null  int64         
 4   InvoiceDate  1044848 non-null  datetime64[us]
 5   Price        1044848 non-null  float64       
 6   Customer ID  809561 non-null   float64       
 7   Country      1044848 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 63.8 MB


## Column profile vs. data contract

Rows: 1,044,848 (matches extract output)

| Column | Contract | Data | Result |
|---|---|---|---|
| Invoice | text, required | `str` (forced on read), 0 missing | OK, Format rule not yet checked |
| StockCode | text, required | `str` (forced on read), 0 missing | OK, Format rule not yet checked |
| Description | text, optional | `str`, 4,275 missing (0.4 %) | OK |
| Quantity | integer, required | `int64` (inferred), 0 missing | All values are integers. Value rules not yet checked |
| InvoiceDate | timestamp, required | `datetime64` (parsed), 0 missing | All timestamps valid |
| Price | decimal, required | `float64` (inferred), 0 missing | Value rules not yet checked |
| Customer ID | text, optional | `float64`, 235,287 missing (22.5 %) | Float due to NaN, FIX in transform. Nearly a quarter of sales has no customer |
| Country | text, optional | `str`, 0 missing | OK |

In [5]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.044848e+06,1044848,1.044848e+06,809561.000000
mean,9.993649e+00,2011-01-03 11:44:15.916841,4.590546e+00,15319.760422
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-05 11:11:00,1.250000e+00,13969.000000
50%,3.000000e+00,2010-12-08 16:34:00,2.100000e+00,15248.000000
75%,1.000000e+01,2011-07-27 13:42:00,4.130000e+00,16792.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.742185e+02,NaN,1.217042e+02,1695.812057


In [6]:
# Count of rows where Quantity is 0
(df["Quantity"] == 0).sum()


np.int64(0)

In [7]:
# Rows where Price is negative
df[df["Price"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
802921,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
802922,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [8]:
# Rows where Quantity is 80995 or -80995
df[df["Quantity"].abs() == 80995]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1043359,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
1043360,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


In [9]:
df[df["Invoice"].str.startswith("A")]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
802920,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
802921,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
802922,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


| Check | Contract | Finding | Result |
|---|---|---|---|
| Quantity = 0 | ≠ 0 → REJECT | 0 rows | OK |
| Quantity extremes | negative only on `C` invoices | ±80,995 = order `581483` cancelled 12 min later by `C581484` (£168,469.60) | OK, Cancellations must be kept, otherwise sales are overstated |
| Cancellation link | — | Cancellation invoice does not reference the original invoice number | OPEN: link can only be inferred from customer + product + quantity |
| Price < 0 | < 0 → REJECT | 5 rows, all `Adjust bad debt` |  New row type, see below |
| `A` invoices | not documented | 6 rows: `Invoice` prefix `A`, `StockCode` `B`, no customer. Net −£147,614.08 |  Undocumented. Accounting adjustments, not sales → REJECT with reason |
| InvoiceDate range | not in the future | 2009-12-01 – 2011-12-09 | OK |
| Customer ID range | 5 digits | 12346 – 18287 | OK |
| Description with comma | — | e.g. `PAPER CRAFT , LITTLE BIRDIE` | OK, Correctly quoted in CSV |

## Keys, duplicates and group consistency

In [10]:
# Exact duplicates: extra rows (keep="first")
df.duplicated().sum()

np.int64(11812)

In [11]:
# Key duplicates (Invoice + StockCode): extra rows
df.duplicated(subset=["Invoice", "StockCode"]).sum()

np.int64(23424)

In [12]:
def count_inconsistent_groups(df, group_col, value_col, dropna=True):
    """Return how many groups have more than one distinct value in value_col."""
    distinct_per_group = df.groupby(group_col)[value_col].nunique(dropna=dropna)
    return int((distinct_per_group > 1).sum())

In [13]:
print("Invoices with >1 Customer ID (incl. NaN):",
      count_inconsistent_groups(df, "Invoice", "Customer ID", dropna=False))
print("Invoices with >1 Country:",
      count_inconsistent_groups(df, "Invoice", "Country"))
print("Invoices with >1 InvoiceDate:",
      count_inconsistent_groups(df, "Invoice", "InvoiceDate"))
print("StockCodes with >1 Description:",
      count_inconsistent_groups(df, "StockCode", "Description"))

Invoices with >1 Customer ID (incl. NaN): 0
Invoices with >1 Country: 0
Invoices with >1 InvoiceDate: 83
StockCodes with >1 Description: 1232


In [14]:
invoice_time_range = df.groupby("Invoice")["InvoiceDate"].agg(["min", "max"])
invoice_time_range["spread"] = invoice_time_range["max"] - invoice_time_range["min"]
print(invoice_time_range.head())

                        min                 max spread
Invoice                                               
489434  2009-12-01 07:45:00 2009-12-01 07:45:00 0 days
489435  2009-12-01 07:46:00 2009-12-01 07:46:00 0 days
489436  2009-12-01 09:06:00 2009-12-01 09:06:00 0 days
489437  2009-12-01 09:08:00 2009-12-01 09:08:00 0 days
489438  2009-12-01 09:24:00 2009-12-01 09:24:00 0 days


In [15]:
multi_time_invoices = invoice_time_range[invoice_time_range["spread"] > pd.Timedelta(0)]
print(len(multi_time_invoices), "invoices with multiple InvoiceDate values")

83 invoices with multiple InvoiceDate values


In [16]:
multi_time_invoices["spread"].value_counts()

spread
0 days 00:01:00    80
0 days 00:04:00     1
0 days 00:02:00     1
0 days 00:09:00     1
Name: count, dtype: int64

In [17]:
descriptions_per_product = (
    df.groupby("StockCode")["Description"]
    .nunique()
    .sort_values(ascending=False)
)
descriptions_per_product.head(5)

StockCode
20713    9
22734    7
23084    7
22423    7
21181    7
Name: Description, dtype: int64

In [18]:
# All descriptions of the product with the most variants
top_product = descriptions_per_product.index[0]
df[df["StockCode"] == top_product]["Description"].value_counts()

Description
JUMBO BAG OWLS                  1346
missing                            1
wrongly marked. 23343 in box       1
wrongly coded-23343                1
found                              1
Found                              1
wrongly marked 23343               1
Marked as 23343                    1
wrongly coded 23343                1
Name: count, dtype: int64

## Keys, duplicates and group consistency: findings

| Check | Contract | Finding | Result |
|---|---|---|---|
| Exact duplicates | WARN, keep | 11,812 extra rows (1.1 %). Naive sheet concat had 34,335 → extract removed 22,523 → every removed overlap row was an exact duplicate | OK, Extract fix verified |
| Key duplicates (Invoice + StockCode) | Allowed | 23,424 extra rows (2.2 %). 11,612 of them differ in another field (e.g. quantity) | OK, Same product added more than once |
| Customer ID per invoice | Same on all rows → WARN | 0 invoices, also counting NaN | OK, Missing customer is invoice-level, which supports "NULL = unknown customer" |
| Country per invoice | Same on all rows → WARN | 0 invoices | OK |
| InvoiceDate per invoice | Same on all rows → WARN | 83 invoices (0.15 %). 80 differ by 1 min, max 9 min. Large invoices, none cross a date boundary | Technical, no reporting impact. Use earliest timestamp |
| Description per StockCode | Same on all rows → WARN | 1,232 products. E.g. `20713`: "JUMBO BAG OWLS" ×1,346 + lowercase stock notes ("missing", "found", "wrongly coded 23343") | Description also holds free-text stock adjustment notes. Rule too noisy → refine in contract v0.2 |

## Business rules (contract section 6)

In [26]:
is_canceled = df["Invoice"].str.startswith("C")
negative_without_c = df[(df["Quantity"] < 0) & (~is_canceled)]
positive_on_c = df[(df["Quantity"] > 0) & (is_canceled)]

print(f"Negative quantities without cancellation: {len(negative_without_c)}")
print(f"Positive quantities with cancellation: {len(positive_on_c)}")

Negative quantities without cancellation: 3393
Positive quantities with cancellation: 1


In [27]:

print(f"price values: {negative_without_c['Price'].unique()[:10]}")
print("Missing Customer ID:", int(negative_without_c["Customer ID"].isna().sum()))
negative_without_c["Description"].value_counts(dropna=False).head(10)

price values: [0.]
Missing Customer ID: 3393


Description
NaN                      2633
check                     121
damages                    83
?                          81
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Name: count, dtype: int64

In [28]:
positive_on_c

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


In [29]:
# Invoice format: 6 digits, optionally prefixed with C
valid_invoice = df["Invoice"].str.fullmatch(r"C?\d{6}")
print("Invoices not matching format:", int((~valid_invoice).sum()))
df[~valid_invoice]["Invoice"].str[0].value_counts()

Invoices not matching format: 6


Invoice
A    6
Name: count, dtype: int64

In [30]:
# StockCode format: 5 digits, optionally followed by one letter
valid_stockcode = df["StockCode"].str.fullmatch(r"\d{5}[A-Za-z]?")
print("Rows with non-standard StockCode:", int((~valid_stockcode).sum()))
print("Distinct non-standard codes:", df[~valid_stockcode]["StockCode"].nunique())
df[~valid_stockcode]["StockCode"].value_counts().head(20)

Rows with non-standard StockCode: 7342
Distinct non-standard codes: 68


StockCode
POST            2086
DOT             1425
M               1398
15056BL          906
C2               277
79323LP          232
D                173
79323GR          123
S                102
BANK CHARGES     100
15056bl           87
ADJUST            67
AMAZONFEE         36
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
gift_0001_10      16
Name: count, dtype: int64

In [31]:
# Zero-price rows: what are they?
zero_price = df[df["Price"] == 0]
print("Zero-price rows:", len(zero_price))
print("  without customer:", int(zero_price["Customer ID"].isna().sum()))
print("  with negative quantity:", int((zero_price["Quantity"] < 0).sum()))
zero_price["Description"].value_counts(dropna=False).head(8)

Zero-price rows: 6024
  without customer: 5954
  with negative quantity: 3393


Description
NaN                      4275
check                     160
?                          90
damages                    83
damaged                    81
found                      28
missing                    27
sold as set on dotcom      20
Name: count, dtype: int64